# Badge 1: Data Warehousing Workshop: Lesson 10: Data Storage Structures

LIBRARY_CARD_CATALOG: BOOK, AUTHOR, a sequence, and the BOOK_TO_AUTHOR bridge table.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, which are swapped for `BADGE1_DWW_ADMIN` (ACCOUNTADMIN), `BADGE1_DWW_ROLE` (SYSADMIN) and `BADGE1_DWW_WH` (COMPUTE_WH).
- Cells marked **SQL equivalent** replace a UI-only step; **Derived solution** marks challenge-lab answers the course does not give.
- DORA checks are not run here; see `dora/dora_tests.ipynb`.
- Files the course has learners download are on the `@UTIL_DB.PUBLIC.BADGE1_DOWNLOADS` stage, which stands in for the learner's Downloads folder.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE1_DWW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE1_DWW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE1_DWW_WH"      # stands in for the course's COMPUTE_WH


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result

In [ ]:
%%sql -r session_context
use role {{ADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
select current_role(), current_warehouse();

## Page: 🎭 Data Storage: A Short History

Video and knowledge checks only.

## Page: 🥋 Create the Library Card Catalog DB

In [ ]:
%%sql -r create_library_db
use role {{SYSADMIN_ROLE}};

// Create a new database and set the context to use the new database
create database library_card_catalog comment = 'DWW Lesson 10 ';

//Set the worksheet context to use the new database
use database library_card_catalog;

## Page: 🎭 Data Storage: Entities and Relationships / From ERDs to Tables

Videos and knowledge checks only.

## Page: 🥋 Create and Fill the Books Table

In [ ]:
%%sql -r create_fill_book
use database library_card_catalog;
use role {{SYSADMIN_ROLE}};

// Create the book table and use AUTOINCREMENT to generate a UID for each new row

create or replace table book
( book_uid number autoincrement
 , title varchar(50)
 , year_published number(4,0)
);

// Insert records into the book table
// You don't have to list anything for the
// BOOK_UID field because the AUTOINCREMENT property 
// will take care of it for you

insert into book(title, year_published)
values
 ('Food',2001)
,('Food',2006)
,('Food',2008)
,('Food',2016)
,('Food',2015);

// Check your table. Does each row have a unique id? 
select * from book;

## Page: 🥋 Create the Author Table

In [ ]:
%%sql -r create_author
use role {{SYSADMIN_ROLE}};
use database library_card_catalog;
// Create Author table
create or replace table author (
   author_uid number 
  ,first_name varchar(50)
  ,middle_name varchar(50)
  ,last_name varchar(50)
);

// Insert the first two authors into the Author table
insert into author(author_uid, first_name, middle_name, last_name)  
values
(1, 'Fiona', '','Macdonald')
,(2, 'Gian','Paulo','Faleschini');

// Look at your table with its new rows
select * 
from author;

> ⚠️ **Audit note:** The Create the Author Table code has no `use` statements; it relies on the context set on the previous page. The `use` lines above make that explicit.

## Page: 🥋 Create a Sequence

> **Course:** UI steps (DWW_156, DWW_156a): Catalog → Database Explorer → LIBRARY_CARD_CATALOG → PUBLIC → **Create** → **Sequence**; the dialog generates the SQL below. NOTE: If you do not include the word ORDER, your values will skip by 100 each time.

In [ ]:
%%sql -r create_sequence
-- SQL from the Create Sequence dialog (DWW_156a)
use role {{SYSADMIN_ROLE}};
use schema library_card_catalog.public;
create sequence SEQ_AUTHOR_UID
    start = 1,
    increment = 1,
    ORDER,
    comment = 'Use this to fill in AUTHOR_UID';

> **Course:** View the Sequence Object: notice that the Next Value column says “1” (DWW_159).

In [ ]:
%%sql -r show_sequence_1
show sequences;

In [ ]:
%%sql -r query_sequence
use role {{SYSADMIN_ROLE}};

//See how the nextval function works
select seq_author_uid.nextval;

> **Course:** Run the query a few more times and see how the next_value changes. When the show sequences result shows next_value = 4, stop.

In [ ]:
%%sql -r query_sequence_2
select seq_author_uid.nextval;

In [ ]:
%%sql -r query_sequence_3
select seq_author_uid.nextval;

In [ ]:
%%sql -r show_sequence_4
show sequences;

> **Course:** Double It Up! You can use nextval more than once per query (DWW_162).

In [ ]:
%%sql -r double_it_up
SELECT SEQ_AUTHOR_UID.nextval, SEQ_AUTHOR_UID.nextval;

In [ ]:
%%sql -r show_sequence_after
show sequences;

## Page: 🥋 Reset the Sequence & Add Rows to the Author Table

In [ ]:
%%sql -r reset_sequence
use role {{SYSADMIN_ROLE}};

//Drop and recreate the counter (sequence) so that it starts at 3 
// then we'll add the other author records to our author table
create or replace sequence library_card_catalog.public.seq_author_uid
start = 3 
increment = 1 
ORDER
comment = 'Use this to fill in the AUTHOR_UID every time you add a row';

In [ ]:
%%sql -r add_authors
//Add the remaining author records and use the nextval function instead 
//of putting in the numbers
insert into author(author_uid,first_name, middle_name, last_name) 
values
(seq_author_uid.nextval, 'Laura', 'K','Egendorf')
,(seq_author_uid.nextval, 'Jan', '','Grover')
,(seq_author_uid.nextval, 'Jennifer', '','Clapp')
,(seq_author_uid.nextval, 'Kathleen', '','Petelinsek');

> ⚠️ **Audit note:** The course's code block for this step is split into two cells unchanged: Workspace notebooks do not treat `//` as a comment, so the apostrophe in the comment *then we'll add…* makes the block fail when run as one cell. The same block runs fine in a Snowsight worksheet.

## Page: 🎭 Data Storage: Mapping the Many-to-Many

Video and knowledge checks only.

## Page: 🥋 Create the Book_To_AUTHOR Table

In [ ]:
%%sql -r create_book_to_author
use database library_card_catalog;
use role {{SYSADMIN_ROLE}};


// Create the relationships table
// this is sometimes called a "Many-to-Many table"
create table book_to_author
( book_uid number
  ,author_uid number
);

//Insert rows of the known relationships
insert into book_to_author(book_uid, author_uid)
values
 (1,1)  // This row links the 2001 book to Fiona Macdonald
,(1,2)  // This row links the 2001 book to Gian Paulo Faleschini
,(2,3)  // Links 2006 book to Laura K Egendorf
,(3,4)  // Links 2008 book to Jan Grover
,(4,5)  // Links 2016 book to Jennifer Clapp
,(5,6); // Links 2015 book to Kathleen Petelinsek


//Check your work by joining the 3 tables together
//You should get 1 row for every author
select * 
from book_to_author ba 
join author a 
on ba.author_uid = a.author_uid 
join book b 
on b.book_uid=ba.book_uid; 

### 🤖 DORA check DWW15 goes here
The course has the learner run **DWW15** (3NF DB was Created, expects 6) at this point.
It is in `dora/dora_tests.ipynb`, section **DWW15**, as a local check plus the exact DORA code.